# Web arayüzü demosu (Colab)

Bu defter Gradio web arayüzünü (`app.py`) Colab'da çalıştırır ve genel geçici bir bağlantı verir.

1. **0. Ayarlar** ve **1. Kurulum** hücrelerini çalıştırın. Önceden yüklü paketler değişirse hücre çalışma zamanını yeniden başlatmanızı ister; yeniden başlattıktan sonra 0. ve 1. bölümü tekrar çalıştırın (kurulum atlanır).
2. **2. Testler** hızlı arayüz testlerini çalıştırır.
3. **3. Arayüz** hücresi `python app.py --share` çalıştırır; çıktıdaki `https://….gradio.live` bağlantısını açın. Durdurmak için hücreyi durdurun.

GPU şart değildir; varsa ECAPA otomatik olarak GPU'yu kullanır (*Çalışma zamanı → Çalışma zamanı türünü değiştir*). Modeller ilk analizde bir kez yüklenir; ECAPA modeli Drive'daki `Hafta4/modeller/ecapa` önbelleğinden okunur, yoksa oraya indirilir.

## 0. Ayarlar

In [ ]:
import os, sys, json, subprocess

KOK = "/content/drive/MyDrive/BilgisayarTasarim_Proje68"
ECAPA_ONBELLEK = f"{KOK}/Hafta4/modeller/ecapa"     # ECAPA modeli bir kez indirilir
REPO_URL = "https://github.com/HekimSefkan/Konusmaci-Tespiti.git"
DAL = "sefkan/arayuz"                                # PR birleşince "main"
REPO = "/content/Konusmaci-Tespiti"

os.environ["KT_MODEL_ONBELLEK"] = ECAPA_ONBELLEK
os.environ["KT_IZINLI_DIZIN"] = KOK                  # "Sunucudaki ses yolu" yalnızca bu klasörün altını açar
os.environ["KT_GECMIS_DIZIN"] = f"{KOK}/Hafta7/gecmis"  # analiz geçmişi

### Şifre

`--share` bağlantısı herkese açıktır. Aşağıdaki hücre kullanıcı adı ve şifreyi sorar ve yalnızca bu oturumun ortam değişkenlerine yazar; deftere kaydedilmez. Şifresiz başlatmak için hücreyi atlayın (uygulama uyarı yazar).

In [ ]:
from getpass import getpass

os.environ["KT_KULLANICI"] = input("Kullanıcı adı: ").strip() or "grup68"
os.environ["KT_SIFRE"] = getpass("Şifre: ")
print("Şifre koruması:", "açık" if os.environ["KT_SIFRE"] else "kapalı (şifre boş)")

## 1. Kurulum

Drive bağlanır, repo klonlanır ya da uzak daldaki son sürüme güncellenir (`git fetch` + `git reset --hard`). `requirements.txt`'teki `torch`, `torchaudio` ve CPU `--extra-index-url` satırları atlanır; Colab'ın torch'u kullanılır (Hafta 4 defterindeki gibi). torchaudio ≥ 2.9 ile speechbrain 1.0.3 uyumsuzluğu `src/uyumluluk.py` yamasıyla giderilir.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "--branch", DAL, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", DAL], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", DAL], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{DAL}"], check=True)
print("Repo:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

PAKETLER = ["numpy", "torch", "torchaudio", "speechbrain", "silero-vad", "librosa", "soundfile",
            "pyannote.metrics", "huggingface_hub", "matplotlib", "gradio", "plotly"]


def surumler():
    """Paket sürümleri, ayrı bir süreçte (bu çekirdekte yüklü eski sürümler karışmasın)."""
    k = ("import importlib.metadata as m, json\nd = {}\n"
         f"for p in {PAKETLER!r}:\n"
         "    try: d[p] = m.version(p)\n    except Exception: d[p] = None\nprint(json.dumps(d))")
    return json.loads(subprocess.run([sys.executable, "-c", k], capture_output=True, text=True).stdout)


ISARET = "/content/.kt_arayuz_kurulum_tamam"
if os.path.isfile(ISARET):
    print("Kurulum daha önce yapılmış; atlanıyor.")
else:
    once = surumler()
    secilen = []
    for s in open(f"{REPO}/requirements.txt", encoding="utf-8"):
        s = s.strip()
        if not s or s.startswith("#") or s.startswith("--extra-index-url"):
            continue
        if s.startswith(("torch==", "torchaudio==")):  # Colab'ın torch'u kullanılır
            continue
        secilen.append(s)
    open("/content/gereksinimler_arayuz.txt", "w").write("\n".join(secilen) + "\n")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/gereksinimler_arayuz.txt"], check=True)
    sonra = surumler()
    degisen = {p: (once[p], sonra[p]) for p in PAKETLER if once[p] != sonra[p]}
    json.dump({"surumler": sonra}, open(ISARET, "w"))
    onceden_yuklu = [p for p, (a, b) in degisen.items() if a is not None]
    if onceden_yuklu:
        print("!" * 78)
        print("!!  ÇALIŞMA ZAMANINI YENİDEN BAŞLATIN: Çalışma zamanı > Oturumu yeniden başlat")
        print("!!  Önceden yüklü paketler değişti:", ", ".join(onceden_yuklu))
        print("!!  Sonra 0. ve 1. bölümü tekrar çalıştırın (kurulum atlanır).")
        print("!" * 78)

print("Sürümler:")
for p, v in surumler().items():
    print(f"  {p:<18} {v}")
r = subprocess.run([sys.executable, "-c",
                    "import torch; print('CUDA:', torch.cuda.is_available()); "
                    "from src.uyumluluk import speechbrain_hazirla; print('uyumluluk yaması:', speechbrain_hazirla() or 'gerekmedi'); "
                    "from speechbrain.inference.speaker import EncoderClassifier; print('speechbrain: içe aktarıldı')"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print(*r.stderr.strip().splitlines()[-5:], sep="\n")
    print("UYARI: speechbrain içe aktarılamadı; arayüz ECAPA ile çalışmaz.")

## 2. Testler

Arayüz mantığı ve sözleşme testleri (model gerektirmez).

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_web_arayuz.py",
                    "tests/test_arayuz_gelismis.py", "tests/test_arayuz.py"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])

## 3. Arayüz

`python app.py --share` çalışır; çıktıdaki **public URL** (`https://….gradio.live`) bağlantısını açın. Bağlantı geçicidir ve bu hücre çalıştığı sürece geçerlidir. Durdurmak için hücreyi durdurun (■).

"Sunucudaki ses yolu" alanına proje klasörüne göre yol yazılabilir, ör. `Hafta2/ornek_ses/sldwj_000.0-090.0.wav` (aynı adlı RTTM `data/sample/`'da olduğu için referans otomatik gelir). Analizler `Hafta7/gecmis/` altına kaydedilir. Bağlantıya şifreyle girin.

In [ ]:
os.chdir(REPO)
!python app.py --share